# Count September's lost campaigns

*Data Analysis in Python · Topic 3.1*

The campaign report is about to be rebuilt, and someone has to decide what happens to the rows without a campaign. Answer for September 2024's export, the whole file.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell.
2. In the first cell marked ✏️, read the file into the `september_raw` table with its dates as dates, and create `missing_by_column`: the number of missing values in each column.
3. In the second cell marked ✏️, look at the `session_campaign` column's values with `value_counts(dropna=False)`, and create three numbers:
   - `nan_rows`: the rows whose campaign is `NaN`
   - `not_set_rows`: the rows whose campaign is `(not set)`
   - `dash_rows`: the rows whose campaign is `-`
4. In the third cell marked ✏️, read the file again into the `september` table, this time with `(not set)` and `-` read as missing in `session_campaign` only. Create:
   - `lost_campaigns`: the number of rows whose campaign is now missing
   - `lost_sessions`: the total sessions in those rows, which is what `dropna()` would throw away
5. In the last cell marked ✏️, replace the `september` table's missing campaigns with the label `"(not set)"`. Don't remove any rows.
6. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `missing_by_column` is 65 for `session_campaign` and 0 for every other column. `nan_rows` is `65`, `not_set_rows` is `122` and `dash_rows` is `20`. `lost_campaigns` is `207` and `lost_sessions` is `820`. After the last step, the `september` table still has 1,780 rows and 99,726 sessions, no missing campaigns, and 207 rows labeled `(not set)`.

## Parameters

In [ ]:
import pandas as pd

FILE_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/sessions_export_2024_09.csv"

## As pandas reads it

In [ ]:
september_raw = pd.read_csv(FILE_URL, parse_dates=["date"])
missing_by_column = september_raw.isna().sum()
missing_by_column

In [ ]:
campaigns = september_raw["session_campaign"]
nan_rows = campaigns.isna().sum()
not_set_rows = (campaigns == "(not set)").sum()
dash_rows = (campaigns == "-").sum()
campaigns.value_counts(dropna=False)

## Every lost campaign as missing

In [ ]:
september = pd.read_csv(FILE_URL, parse_dates=["date"], na_values={"session_campaign": ["(not set)", "-"]})
lost = september["session_campaign"].isna()
lost_campaigns = lost.sum()
lost_sessions = september.loc[lost, "sessions"].sum()
lost_campaigns, lost_sessions

In [ ]:
september["session_campaign"] = september["session_campaign"].fillna("(not set)")
len(september), september["sessions"].sum()

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    missing = found.get("missing_by_column")
    if isinstance(missing, pd.Series) and "session_campaign" in missing.index and missing["session_campaign"] == 65 and missing.sum() == 65:
        results.append((True, "missing_by_column: 65 missing campaigns, and no gaps in any other column."))
    elif isinstance(missing, pd.Series) and missing.sum() == 1780 * 7 - 65:
        results.append((False, "missing_by_column counts the values that aren't missing. Use isna(), not notna()."))
    elif isinstance(missing, pd.Series) and "session_campaign" in missing.index and missing["session_campaign"] == 207:
        results.append((False, "missing_by_column counts 207 missing campaigns: september_raw was read with na_values. Read it the default way, the way pandas sees the file on its own."))
    else:
        results.append((False, "missing_by_column should be september_raw.isna().sum(): one count per column."))

    counts = (found.get("nan_rows"), found.get("not_set_rows"), found.get("dash_rows"))
    if tables := [name for name, value in zip(["nan_rows", "not_set_rows", "dash_rows"], counts) if isinstance(value, (pd.DataFrame, pd.Series))]:
        results.append((False, f"{', '.join(tables)} should be numbers, not tables of rows. Count with .sum() on a mask: (september_raw[\"session_campaign\"] == \"-\").sum()."))
    elif counts == (65, 122, 20):
        results.append((True, "Three spellings of a lost campaign: 65 NaN, 122 (not set) and 20 -."))
    elif counts[0] == 0:
        results.append((False, "nan_rows is 0. value_counts() leaves NaN out unless you pass dropna=False; isna().sum() counts it directly."))
    else:
        results.append((False, f"nan_rows, not_set_rows and dash_rows should be 65, 122 and 20, but they are {counts}. Count in september_raw: (september_raw[\"session_campaign\"] == \"(not set)\").sum()."))

    september = found.get("september")
    lost = found.get("lost_campaigns")
    lost_sessions = found.get("lost_sessions")
    if isinstance(lost, (pd.DataFrame, pd.Series)):
        results.append((False, f"lost_campaigns is a {type(lost).__name__}, not a number. Count the rows instead of keeping them: september[\"session_campaign\"].isna().sum()."))
    elif lost == 207:
        results.append((True, "207 rows have a lost campaign, in any spelling."))
    elif lost == 187:
        results.append((False, "lost_campaigns is 187: the 20 rows with - aren't missing yet. Add \"-\" to the list in na_values."))
    elif lost == 85:
        results.append((False, "lost_campaigns is 85: the 122 rows with (not set) aren't missing yet. Add \"(not set)\" to the list in na_values."))
    elif lost == 65:
        results.append((False, "lost_campaigns is 65, only the empty cells. Read september with na_values={\"session_campaign\": [\"(not set)\", \"-\"]}."))
    else:
        results.append((False, f"lost_campaigns should be september[\"session_campaign\"].isna().sum() after reading with na_values, before labeling. It is {lost!r}."))
    if isinstance(lost_sessions, (pd.DataFrame, pd.Series)):
        results.append((False, f"lost_sessions is a {type(lost_sessions).__name__}, not a number. Total one column of the rows: september.loc[mask, \"sessions\"].sum()."))
    elif lost_sessions == 820:
        results.append((True, "820 sessions have a lost campaign: dropna() would throw them away, and every total with them."))
    elif lost_sessions == 208:
        results.append((False, "lost_sessions is 208, the sessions of the empty cells only. Total the sessions of every row whose campaign is missing in september."))
    elif lost_sessions == 207:
        results.append((False, "lost_sessions is the number of rows. Total their sessions: september.loc[mask, \"sessions\"].sum()."))
    else:
        results.append((False, f"lost_sessions should be the total sessions of the rows whose campaign is missing. It is {lost_sessions!r}."))

    if not isinstance(september, pd.DataFrame):
        results.append((False, "september isn't a DataFrame yet. Read the file again with na_values."))
    elif len(september) == 1573:
        results.append((False, "september has 1,573 rows: dropna() removed 207 rows and 820 sessions. Label the missing campaigns with fillna(\"(not set)\") instead."))
    elif len(september) != 1780:
        results.append((False, f"september has {len(september):,} rows, but the file has 1,780. Labeling shouldn't remove any rows."))
    elif september["session_campaign"].isna().any():
        results.append((False, f"september still has {september['session_campaign'].isna().sum():,} missing campaigns. Assign the result back: september[\"session_campaign\"] = september[\"session_campaign\"].fillna(\"(not set)\")."))
    elif (september["session_campaign"] == "-").any():
        results.append((False, "september still has campaigns spelled -. Read it with na_values, so they're missing before you label them."))
    elif (september["session_campaign"] == "(not set)").sum() == 207 and september["sessions"].sum() == 99726:
        results.append((True, "september keeps all 1,780 rows and 99,726 sessions, with 207 campaigns labeled (not set)."))
    else:
        results.append((False, f"september has {(september['session_campaign'] == '(not set)').sum():,} rows labeled (not set), but it should have 207. Use fillna(\"(not set)\")."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
september_raw = pd.read_csv(FILE_URL, parse_dates=["date"])
missing_by_column = september_raw.isna().sum()
missing_by_column
```

```python
campaigns = september_raw["session_campaign"]
nan_rows = campaigns.isna().sum()
not_set_rows = (campaigns == "(not set)").sum()
dash_rows = (campaigns == "-").sum()
campaigns.value_counts(dropna=False)
```

```python
september = pd.read_csv(FILE_URL, parse_dates=["date"], na_values={"session_campaign": ["(not set)", "-"]})
lost = september["session_campaign"].isna()
lost_campaigns = lost.sum()
lost_sessions = september.loc[lost, "sessions"].sum()
lost_campaigns, lost_sessions
```

```python
september["session_campaign"] = september["session_campaign"].fillna("(not set)")
len(september), september["sessions"].sum()
```

</details>